In [0]:
# !pip install -q tensorflow-gpu==1.13.0rc0 opencv-python
!rm -rf *
!rm -rf .gi*

In [0]:
#from fileutils import *

#from fileutils import download_file, extract_file_from_tar, exec_cmd
from PIL import Image
from matplotlib import pyplot as plt
from io import StringIO
from collections import defaultdict
from distutils.version import StrictVersion
import cv2
import time
import zipfile
import tensorflow as tf
import tarfile
import sys
import six.moves.urllib as urllib
import os
import numpy as np

sys.path.append('..')


class ssd:

    def __init__(self, prepared=False):
        self.name = 'SSD'
        self.env_dir = 'env_' + self.name
        self.prepared = prepared
        self.base_url = 'http://download.tensorflow.org/models/object_detection/'
        self.available_models = ['ssd_mobilenet_v1_coco_2017_11_17',
                                 'ssd_resnet50_v1_fpn_shared_box_predictor_640x640_coco14_sync_2018_07_03', 'faster_rcnn_nas_coco_2018_01_28']
        self.model_name = self.available_models[-1]
        self.model_file = self.model_name + '.tar.gz'
        self.frozen_graph_name = 'frozen_inference_graph.pb'
        self.frozen_graph = self.model_name + '/' + self.frozen_graph_name

        if StrictVersion(tf.__version__.split('-')[0]) < StrictVersion('1.9'):
            raise ImportError(
                'Please upgrade your TensorFlow installation to v1.9.* or later!')
        print('Using OpenCV version %r and Tensorflow version %r' %
              (cv2.__version__, tf.__version__))
        
    def prepare_env(self):
        env_dir = self.env_dir
        print('preparing environment')
        # env preparation since this notebook is being run as standalone
        #exec_cmd('rm -rf * ')
        exec_cmd('mkdir '+env_dir+'')
        exec_cmd(
            'git clone https://github.com/tensorflow/models.git '+env_dir+'/md --recursive')
        exec_cmd(
            'git clone https://github.com/cocodataset/cocoapi.git '+env_dir+'/cocoapi')
        exec_cmd(
            'cd '+env_dir+'/cocoapi/PythonAPI && make && cp -rv pycocotools ../../md/research/')
        exec_cmd(
            'cd '+env_dir+'/md/research && protoc object_detection/protos/*.proto --python_out=.')
        exec_cmd('cd '+env_dir+'/md/research && python setup.py install')
        exec_cmd('cd '+env_dir+'/md/research/slim && python setup.py install')
        exec_cmd(
            'cd '+env_dir+'/md/research && python object_detection/builders/model_builder_test.py')
        exec_cmd('mv -v '+env_dir+'/md/research/* '+env_dir+'/')
        #exec_cmd('mv '+env_dir+'/md/research/object_detection ./')
        exec_cmd('mv -v '+env_dir+'/md/research/setup.py '+env_dir+'/')
        exec_cmd('ln -s '+env_dir+'/object_detection/data data')
        exec_cmd('rm -rf '+env_dir+'/md')
        exec_cmd('ls '+env_dir+'/')
        exec_cmd('python '+env_dir +
                 '/object_detection/builders/model_builder_test.py')
        
    def prepare(self, prepared=None,count=1):
        prepared = self.prepared if prepared is None else prepared
        if not prepared:
            self.prepare_env()
        else:
            print('No need to prepare environment')
        self.prepared = self.import_utils()
        self.lalbels_file = os.path.join('data', 'mscoco_label_map.pbtxt')
        download_file(self.base_url + self.model_file, self.model_file)
        extract_file_from_tar(self.model_file, self.frozen_graph_name)
        self.detection_graph = self.get_detection_graph(self.frozen_graph)
        self.category_index = label_map_util.create_category_index_from_labelmap(
            self.lalbels_file, use_display_name=True)

    def import_utils(self):
        try:
            sys.path.append(".")
            sys.path.append("..")
            sys.path.append(self.env_dir)
            sys.path.append(self.env_dir+'/object_detection')
            global utils_ops, utils, label_map_util, visualization_utils, vis_util
            from object_detection.utils import ops as utils_ops
            from object_detection import utils
            from utils import label_map_util
            from utils import visualization_utils as vis_util
            print('Done importing utils')
            
            exec_cmd('cp -rv object_detection/* ./')    
        except Exception as e:
            print('\n\nError importing from environment from '+self.env_dir+': ', e)
            return False
        return True


    def detect(self, image, opfile, class_labels_to_filter, detection_graph=None, category_index=None, visualize=False):
        print(' '+opfile.split('/')[-1], end=' ')
        detection_graph = self.detection_graph if not detection_graph else detection_graph
        category_index = self.category_index if not category_index else category_index
        image_np = image
        # Actual detection.
        output_dict = self.run_inference_for_single_image(
            image_np, detection_graph)
        # Visualization of the results of a detection.
        labels_detected = [category_index[obj]['name']
                           for obj in output_dict['detection_classes']]
        labels_matched = set(labels_detected) & set(
            class_labels_to_filter)
        print(labels_matched)
        if visualize:
            vis_util.visualize_boxes_and_labels_on_image_array(image_np, output_dict['detection_boxes'], output_dict['detection_classes'], output_dict['detection_scores'],
                                                               category_index, instance_masks=output_dict.get('detection_masks'), use_normalized_coordinates=True, line_thickness=8)
        cv2.imwrite(opfile, image_np)
        
        bboxes_denormalized = list(map(lambda boxes:self.denormalize(boxes,image_np.shape),output_dict['detection_boxes']))        
        output_dict = self.clean_up_entry({'bounding_boxes':bboxes_denormalized,'labels_detected':labels_detected})
        if output_dict:
          return {'labels_matched': labels_matched, 'output': output_dict}
      
    def denormalize(self,box,imshape):
      im_width,im_height,im_channels = imshape
      ymin, xmin, ymax, xmax = box
      return (xmin * im_width, xmax * im_width, ymin * im_height, ymax * im_height)
    
    def clean_up_entry(self,dct):
      bbk,lbk = 'bounding_boxes','labels_detected'
      bboxes = dct[bbk]
      labels = dct[lbk]
      bboxes = list(map(lambda b:list(map(lambda v:int(v), list(b))),bboxes))
      bboxes = list(filter(lambda x: sum(x)>0,bboxes))
      labels = labels[0:len(bboxes)]
      if labels:
        return {bbk:bboxes,lbk:labels}


    def run_inference_for_single_image(self, image, graph):
        with graph.as_default():
            with tf.Session() as sess:
                # Get handles to input and output tensors
                ops = tf.get_default_graph().get_operations()
                all_tensor_names = {
                    output.name for op in ops for output in op.outputs}
                tensor_dict = {}
                for key in ['num_detections', 'detection_boxes', 'detection_scores', 'detection_classes', 'detection_masks']:
                    tensor_name = key + ':0'
                    if tensor_name in all_tensor_names:
                        tensor_dict[key] = tf.get_default_graph(
                        ).get_tensor_by_name(tensor_name)
                if 'detection_masks' in tensor_dict:
                    # The following processing is only for single image
                    detection_boxes = tf.squeeze(
                        tensor_dict['detection_boxes'], [0])
                    detection_masks = tf.squeeze(
                        tensor_dict['detection_masks'], [0])
                    # Reframe is required to translate mask from box coordinates to image coordinates and fit the image size.
                    real_num_detection = tf.cast(
                        tensor_dict['num_detections'][0], tf.int32)
                    detection_boxes = tf.slice(detection_boxes, [0, 0], [
                                               real_num_detection, -1])
                    detection_masks = tf.slice(detection_masks, [0, 0, 0], [
                                               real_num_detection, -1, -1])
                    detection_masks_reframed = utils_ops.reframe_box_masks_to_image_masks(
                        detection_masks, detection_boxes, image.shape[0], image.shape[1])
                    detection_masks_reframed = tf.cast(
                        tf.greater(detection_masks_reframed, 0.5), tf.uint8)
                    # Follow the convention by adding back the batch dimension
                    tensor_dict['detection_masks'] = tf.expand_dims(
                        detection_masks_reframed, 0)
                image_tensor = tf.get_default_graph().get_tensor_by_name('image_tensor:0')
                # Run inference
                output_dict = sess.run(tensor_dict, feed_dict={
                                       image_tensor: np.expand_dims(image, 0)})
                # all outputs are float32 numpy arrays, so convert types as appropriate
                output_dict['num_detections'] = int(
                    output_dict['num_detections'][0])
                output_dict['detection_classes'] = output_dict['detection_classes'][0].astype(
                    np.uint8)
                output_dict['detection_boxes'] = output_dict['detection_boxes'][0]
                output_dict['detection_scores'] = output_dict['detection_scores'][0]
                if 'detection_masks' in output_dict:
                    output_dict['detection_masks'] = output_dict['detection_masks'][0]
                return output_dict

    def get_detection_graph(self, frozen_graph):
        detection_graph = tf.Graph()
        with detection_graph.as_default():
            od_graph_def = tf.GraphDef()
            with tf.gfile.GFile(frozen_graph, 'rb') as fid:
                serialized_graph = fid.read()
                od_graph_def.ParseFromString(serialized_graph)
                tf.import_graph_def(od_graph_def, name='')
        return detection_graph


In [0]:
##fileutils.py
import os
import six.moves.urllib as urllib

def exec_cmd(cmdstr,echo=True):
  print(os.popen(cmdstr).read() if echo else '',end='')
  
def download_file(url,filename=None):
  fn = (str(' -O '+filename) if filename else ' ')
  exec_cmd('wget -nc '+url+fn)
  return filename if filename else url.split('/')[-1]

# def download_file(url,filename):
#   opener = urllib.request.URLopener()  
#   opener.retrieve(url, filename)
def create_zip(zip_name,file_names):
  print('Creating zip file ',zip_name)
  exec_cmd('zip -r '+zip_name+' '+' '.join(file_names))

def extract_file_from_tar(tar_file,filename_to_ext):
  tar_file = tarfile.open(tar_file)
  for file in tar_file.getmembers():
    file_name = os.path.basename(file.name)
    if filename_to_ext in file_name:
      tar_file.extract(file, os.getcwd())
      
def load_image_into_numpy_array(self,image):
  (im_width, im_height) = image.size
  return np.array(image.getdata()).reshape((im_height, im_width, 3)).astype(np.uint8)

import json
def bbox_annotations(output): # call once per file
  bbox_annotations=[]
  bbk,lbk = 'bounding_boxes','labels_detected'
  tmp = output['output']
  for i in range(len(tmp[bbk])):
    coordinates = [str(val) for val in tmp[bbk][i]]
    pt1 = ','.join(coordinates[:2])
    pt2 = ','.join(coordinates[2:])
    bbox_annotations.append({'classification_label':tmp[lbk][i],'point_2D':[pt1,pt2]})
  return bbox_annotations

def annotate_image(output_item):  # call once per file
  fname, output_dict = output_item
  annotations = {'annotation': {'data_filename': fname, 'data_type': 'image', 'data_annotation': {'bounding_polygon': [], 'bounding_box': ''}}}
  annotations['annotation']['data_annotation']['bounding_box'] = bbox_annotations(output_dict)
  return annotations

def save_as_json(dct,parent_dir='./'): # call once per file
  fname = parent_dir+dct['annotation']['data_filename']
  fname = '.'.join(fname.split('.')[:-1])+'.json'
  try:
    with open(fname,'w+') as f:
      json.dump(dct,f)
    return fname
  except Exception as e:
    print('Error writing to '+fname)
  return ('Failed!',fname)
  
def save_as_annotations(output,opdir='./'):
  print('Saving annotations to file')
  opdir = opdir if opdir[-1]=='/' else opdir+'/' #add trailing / if not present
  annotated_output = list(map(annotate_image,list(output.items())))
  success,failure=[],[]
  for dct in annotated_output:
    fname = save_as_json(dct)
    lst = failure if type(fname) is tuple else success
    lst.append(fname)
  return success,failure
  #result = [{dct['annotation']['data_filename']:save_as_json(dct)} for dct in annotated_output ]
  #return result,list(filter(lambda x: type(tuple(x.items())[0][-1]) is tuple ,gg))

In [0]:
class yolo:
  def __init__(self):
    pass

In [0]:
import os
import sys
# from models.ssd import ssd
# from models.yolo import yolo


class detector:
    models = {'ssd': {'name': 'ssd', 'variant': ''},
              'yolo': {'name': 'yolo', 'variant': 'v2'}}
    datasets = ['voc', 'imagenet', 'coco']
    dataset = datasets[-1]

    def __init__(self,model_name):
        self.models['ssd']['model'] = ssd
        self.models['yolo']['model'] = yolo
        self.model_name = model_name if model_name else 'ssd'
        self.model = self.models[self.model_name]

    def get_model_class(self):
        return self.model['model']

# extractor.py
import cv2
import time

# interval if specified should be in terms of seconds


def extract_frames(detector, input_file, class_labels, interval=None, dest_dir='.'):
    print('Detection Algorithm:%s\nModel loaded: %s\nInput File: %s\nIntervals at which to detect: %r seconds' % (
        detector.name, detector.model_name, input_file, interval))
    interval = interval * 1000  # convert to milliseconds
    cap = cv2.VideoCapture(input_file)
    output = {}  # format: File name : [list of matched labels]
    if (cap.isOpened() == False):
        print("Error opening video file", input_file)
    i, count = 0, 0
    fps = cap.get(cv2.CAP_PROP_FPS)
    input_file_name = '.'.join(input_file.split('.')[:-1])
    total_duration = 0

    if dest_dir[-1] != '/':
        dest_dir = dest_dir + '/'

    while(cap.isOpened()):
        opfname = None
        if interval:
            cap.set(cv2.CAP_PROP_POS_MSEC, interval)
            opfname = input_file_name+'-'+str(int(interval/1000))+'s.jpg'
            interval += interval
        ret, frame = cap.read()
        if ret == True:
            i += 1
            print('Frame: ', i, end=' ')
            #frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            opfname = opfname if opfname else input_file_name + \
                '-'+str(int(i/fps))+':'+str(i % fps)+'.jpg'

            t1 = time.time()
            result = detector.detect(
                frame, dest_dir+opfname, class_labels)
            t2 = time.time()
            duration = t2-t1
            total_duration += duration
            if result:
                result.update({'time': duration})
                output.update({opfname: result})
        else:
          print(ret)
          break

    if i and count:
        print('Overall Processing speed per image', (total_duration)/i)
        print('Frames processed :', i)
    else:
        print('No frames were processed')
    cap.release()
    cv2.destroyAllWindows()
    return output


In [0]:
input_file=download_file('https://github.com/manuhg/masknet/raw/master/input_video.mp4')
class_labels_to_filter_by = ['person']

detector_ = detector('ssd')
detector_model_class = detector_.get_model_class()
detector_model = detector_model_class(True)
detector_model.prepare()
output = extract_frames(detector_model, input_file, class_labels_to_filter_by, interval=4)

Using OpenCV version '3.4.3' and Tensorflow version '1.13.0-rc1'
No need to prepare environment
Done importing utils
Detection Algorithm:SSD
Model loaded: faster_rcnn_nas_coco_2018_01_28
Input File: input_video.mp4
Intervals at which to detect: 4 seconds


In [0]:
opdir = 'detections'
data_dir = opdir+'/data'
meta_dir = opdir+'/meta'
exec_cmd('rm -rf '+opdir)
exec_cmd('mkdir -p '+data_dir)
exec_cmd('mkdir -p '+meta_dir)
json_files,failures = save_as_annotations(output,meta_dir)
if failures:
  print('Failed to write: ',','.join(failures))

exec_cmd('cp '+' '.join(list(output.keys()))+' '+data_dir)
exec_cmd('cp '+' '.join(json_files)+' '+meta_dir)
exec_cmd('zip detections.zip -r '+opdir)